<a href="https://colab.research.google.com/github/filipn05/Final-Year-Project/blob/main/LogoDet_3K_Pretraining_YOLOV11n_(phase_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Methodology

## Phase 1: Class-Agnostic Logo Pretraining — YOLO11n

This notebook documents **Phase 1** of the thesis methodology. The objective is to train the YOLO11n detector to localise logos in a **class-agnostic** setting before adapting the model to the 26 target brand classes in Phase 2.

The LogoDet-3K source categories are therefore **not used as prediction classes**. Every valid logo bounding box is converted to class `0` (`logo`). The source images are shuffled with random seed `42` and divided into an 80/20 training/validation split. Ultralytics is then configured with `fraction=0.3`, so Phase 1 training uses 30% of the generated training split because of the available compute budget.

**Phase 1 configuration represented by this notebook:**
- architecture: YOLO11n, initialised from `yolo11n.yaml`;
- task: single-class object detection (`logo`);
- input resolution: 416 × 416;
- batch size: 64;
- data-loader workers: 2;
- mosaic probability: 0.0;
- training fraction: 0.3;
- intended cumulative training budget: 40 epochs;
- execution: 5-epoch chunks with checkpoints backed up to Google Drive.

### 1. Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis-logodet3k'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print("Drive mounted.")

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

### 2. Dataset acquisition

Downloads LogoDet-3K from Kaggle on first run; subsequent runs restore the verified zip from Drive rather than re-downloading.

In [ ]:
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

if os.path.exists(f'{DRIVE_DATA_DIR}/logodet3k.zip'):
    print("Found existing verified zip on Drive; copying locally.")
    import shutil
    shutil.copy(f'{DRIVE_DATA_DIR}/logodet3k.zip', '/content/logodet3k.zip')
else:
    print("No copy on Drive yet; downloading from Kaggle.")
    !pip install -q kaggle
    !kaggle datasets download -d lyly99/logodet3k

print("Verifying zip integrity...")
!unzip -t /content/logodet3k.zip | tail -3

In [ ]:
!unzip -q /content/logodet3k.zip -d /content/logodet3k_raw
print("Extracted:", len(os.listdir('/content/logodet3k_raw')), "items")

if not os.path.exists(f'{DRIVE_DATA_DIR}/logodet3k.zip'):
    import shutil
    shutil.copy('/content/logodet3k.zip', f'{DRIVE_DATA_DIR}/logodet3k.zip')
    print("Verified zip backed up to Drive.")
else:
    print("Already backed up to Drive.")

### 3. Conversion to YOLO format

Each LogoDet-3K Pascal-VOC-style XML annotation is converted to a single class-agnostic YOLO label (class index 0, "logo"), consistent with the class-agnostic pretraining objective

In [ ]:
import glob
import xml.etree.ElementTree as ET
import random
import shutil
import yaml

LOCAL_RAW_DIR = '/content/logodet3k_raw'
LOCAL_YOLO_DIR = '/content/yolo_dataset'


def convert_box(size, box):
    """Converts a Pascal-VOC (xmin, xmax, ymin, ymax) box to YOLO
    (normalised centre-x, centre-y, width, height) format."""
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)


os.makedirs(f'{LOCAL_YOLO_DIR}/images/train', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/images/val', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/labels/val', exist_ok=True)

image_paths = glob.glob(f'{LOCAL_RAW_DIR}/**/*.jpg', recursive=True)
classes = sorted(set(os.path.basename(os.path.dirname(p)) for p in image_paths))
print(f"Detected {len(classes)} distinct LogoDet-3K logo classes.")

# Fixed seed for a reproducible 80/20 train/val split.
random.seed(42)
random.shuffle(image_paths)
split_idx = int(len(image_paths) * 0.8)

In [ ]:
converted_count = 0
skipped_existing = 0

for idx, img_path in enumerate(image_paths):
    split = 'train' if idx < split_idx else 'val'
    class_name = os.path.basename(os.path.dirname(img_path))
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    unique_name = f"{class_name}_{base_name}"
    xml_path = img_path.replace('.jpg', '.xml')

    out_img = f'{LOCAL_YOLO_DIR}/images/{split}/{unique_name}.jpg'
    out_lbl = f'{LOCAL_YOLO_DIR}/labels/{split}/{unique_name}.txt'

    if os.path.exists(out_img) and os.path.exists(out_lbl):
        skipped_existing += 1
        continue
    if not os.path.exists(xml_path):
        continue

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0:
            continue

        label_lines = []
        for obj in root.iter('object'):
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
            bb = convert_box((w, h), b)
            label_lines.append("0 " + " ".join(f"{a:.6f}" for a in bb))

        if label_lines:
            shutil.copy(img_path, out_img)
            with open(out_lbl, 'w') as f:
                f.write('\n'.join(label_lines))
            converted_count += 1
    except Exception:
        continue

with open(f'{LOCAL_YOLO_DIR}/classes.txt', 'w') as f:
    f.write('\n'.join(classes))

dataset_yaml = {
    'path': LOCAL_YOLO_DIR, 'train': 'images/train', 'val': 'images/val',
    'nc': 1, 'names': {0: 'logo'},
}
with open(f'{LOCAL_YOLO_DIR}/dataset.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Newly converted: {converted_count} | Skipped (already present): {skipped_existing}")
print("images/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/train')))
print("images/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/val')))

In [ ]:
print("Zipping converted dataset for backup...")
!cd /content && zip -qr yolo_dataset_v2.zip yolo_dataset
shutil.copy('/content/yolo_dataset_v2.zip', f'{DRIVE_DATA_DIR}/yolo_dataset_v2.zip')
print("Converted dataset backed up to Drive.")

Sanity Check

In [ ]:
print("images/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/train')))
print("images/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/val')))
print("labels/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/labels/train')))
print("labels/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/labels/val')))

with open(f'{LOCAL_YOLO_DIR}/dataset.yaml') as f:
    print("\n" + f.read())

train_labels = os.listdir(f'{LOCAL_YOLO_DIR}/labels/train')
print(f"\nSample labels (total: {len(train_labels)}):")
for fname in train_labels[:3]:
    with open(f'{LOCAL_YOLO_DIR}/labels/train/{fname}') as f:
        print(fname, '->', f.read().strip()[:150])

### 4. Pretraining

Trained in fixed-size resumable chunks, since Colab sessions are subject to disconnection. Progress is tracked in a counter file and the run folder is backed up to Drive after every chunk; rerunning this cell continues training from wherever it left off, until `TOTAL_EPOCHS` is reached.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
import shutil
from ultralytics import YOLO

CKPT_DIR = f'{DRIVE_DATA_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

RUN_NAME = 'yolo11n_class_agnostic'
LOCAL_RUN_DIR = f'runs/detect/logodet3k_pretrain/{RUN_NAME}'
LOCAL_WEIGHTS = f'{LOCAL_RUN_DIR}/weights/last.pt'
DRIVE_RUN_ZIP = f'{CKPT_DIR}/{RUN_NAME}_run.zip'
PROGRESS_FILE = f'{DRIVE_DATA_DIR}/{RUN_NAME}_epochs_done.txt'

CHUNK_SIZE = 5
TOTAL_EPOCHS = 40

# Restore checkpoint and progress counter from Drive if missing locally
# (e.g. at the start of a new session).
if not os.path.exists(LOCAL_WEIGHTS) and os.path.exists(DRIVE_RUN_ZIP):
    print("Restoring last checkpoint from Drive...")
    shutil.copy(DRIVE_RUN_ZIP, f'{RUN_NAME}_run.zip')
    !unzip -oq {RUN_NAME}_run.zip
    print("Restored.")

completed_epochs = 0
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        completed_epochs = int(f.read().strip())

remaining = TOTAL_EPOCHS - completed_epochs
this_chunk = min(CHUNK_SIZE, remaining)

print(f"{completed_epochs} epoch(s) already done. Training {this_chunk} more "
      f"(target: {completed_epochs + this_chunk}/{TOTAL_EPOCHS}).")

if remaining <= 0:
    print(f"Already reached {TOTAL_EPOCHS} epochs.")
else:
    # Load from the last local checkpoint's weights and train a fresh chunk
    # of epochs, rather than using YOLO's built-in resume=True, which
    # requires the exact original run directory and epoch count to still
    # be present and is therefore fragile across sessions.
    if os.path.exists(LOCAL_WEIGHTS):
        model = YOLO(LOCAL_WEIGHTS)
        print("Continuing from existing local checkpoint.")
    else:
        model = YOLO('yolo11n.yaml')
        print("Starting fine-tuning fresh (no local checkpoint found).")

    results = model.train(
        data=f'{LOCAL_YOLO_DIR}/dataset.yaml',
        epochs=this_chunk,
        patience=100,
        imgsz=416,
        batch=64,
        single_cls=True,   # class-agnostic pretraining (Section 4.4.2)
        workers=2,
        cache=False,
        mosaic=0.0,
        fraction=0.3,      # 30% of LogoDet-3K, due to compute constraints
        project='logodet3k_pretrain',
        name=RUN_NAME,
        exist_ok=True,     # write into the same run folder each chunk
    )

    completed_epochs += this_chunk
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(completed_epochs))

    print("Backing up run and progress counter to Drive...")
    !cd /content && zip -qr {RUN_NAME}_run.zip {LOCAL_RUN_DIR}
    shutil.copy(f'{RUN_NAME}_run.zip', DRIVE_RUN_ZIP)
    print(f"Backed up. {completed_epochs}/{TOTAL_EPOCHS} epochs done. "
          f"Rerun this cell to continue.")

### 5. Final checkpoint

Saves the best-performing epoch's weights under a fixed name, used as the Phase 1 checkpoint for Phase 2 fine-tuning

In [ ]:
shutil.copy(
    f'runs/detect/logodet3k_pretrain/{RUN_NAME}/weights/best.pt',
    f'{DRIVE_DATA_DIR}/yolo11n_pretrained_final.pt'
)
print("Final YOLO11n Phase 1 checkpoint saved as yolo11n_pretrained_final.pt")